# 06 - Error Analysis

**Purpose:** Deep-dive analysis of benchmark errors to identify failure patterns and improvement opportunities.

**Contents:**
- Error rate per chart type
- Worst-performing predictions
- Error category breakdown
- Sample analysis with visualization

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing import List, Dict, Any, Tuple
import warnings

warnings.filterwarnings('ignore')

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Configuration
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'scripts':
    PROJECT_ROOT = PROJECT_ROOT.parent
    
RESULTS_DIR = PROJECT_ROOT / 'data' / 'benchmark' / 'results' / 'runs'

# List available runs
available_runs = []
if RESULTS_DIR.exists():
    for run_dir in RESULTS_DIR.iterdir():
        if run_dir.is_dir() and (run_dir / 'results.json').exists():
            available_runs.append(run_dir.name)

available_runs.sort()
print(f"Available runs ({len(available_runs)}):")
for i, run in enumerate(available_runs):
    print(f"  {i}: {run}")

# Select run (change index as needed)
selected_run_idx = 0 if available_runs else None
if selected_run_idx is not None:
    selected_run = available_runs[selected_run_idx]
    selected_path = RESULTS_DIR / selected_run / 'results.json'
    
    with open(selected_path, 'r') as f:
        run_results = json.load(f)
    
    print(f"\nSelected run: {selected_run}")
else:
    print("No benchmark runs found")
    run_results = {}

In [ ]:
# Error rate analysis per chart type
if run_results and 'per_chart' in run_results:
    per_chart = run_results['per_chart']
    
    chart_types = {}
    for chart_id, metrics in per_chart.items():
        chart_type = metrics.get('chart_type', 'unknown')
        score = metrics.get('score', 0)
        
        if chart_type not in chart_types:
            chart_types[chart_type] = {'scores': [], 'count': 0, 'errors': 0}
        
        chart_types[chart_type]['scores'].append(score)
        chart_types[chart_type]['count'] += 1
        if score < 0.5:  # Consider score < 0.5 as error
            chart_types[chart_type]['errors'] += 1
    
    # Create summary DataFrame
    error_summary = []
    for chart_type, data in chart_types.items():
        error_summary.append({
            'chart_type': chart_type,
            'total': data['count'],
            'errors': data['errors'],
            'error_rate': data['errors'] / data['count'] if data['count'] > 0 else 0,
            'avg_score': np.mean(data['scores'])
        })
    
    df_errors = pd.DataFrame(error_summary).sort_values('error_rate', ascending=False)
    
    print("Error Rate per Chart Type:")
    print("=" * 70)
    print(df_errors.to_string(index=False))
else:
    print("No per-chart metrics available")
    df_errors = pd.DataFrame()

In [ ]:
# Identify top 20 worst predictions (lowest scores)
if run_results and 'per_chart' in run_results:
    per_chart = run_results['per_chart']
    
    worst_predictions = []
    for chart_id, metrics in per_chart.items():
        score = metrics.get('score', 1.0)
        chart_type = metrics.get('chart_type', 'unknown')
        reason = metrics.get('failure_reason', 'Not specified')
        
        worst_predictions.append({
            'chart_id': chart_id,
            'score': score,
            'chart_type': chart_type,
            'failure_reason': reason
        })
    
    df_worst = pd.DataFrame(worst_predictions).nsmallest(20, 'score')
    
    print("Top 20 Worst Predictions (Lowest Scores):")
    print("=" * 80)
    print(df_worst.to_string(index=False))
else:
    print("No per-chart metrics available")
    df_worst = pd.DataFrame()

In [ ]:
# Error category breakdown
if len(df_worst) > 0:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # Chart 1: Error rate by type
    if len(df_errors) > 0:
        df_errors_sorted = df_errors.sort_values('error_rate', ascending=True)
        ax1.barh(df_errors_sorted['chart_type'], df_errors_sorted['error_rate'], color='coral', alpha=0.8)
        ax1.set_xlabel('Error Rate')
        ax1.set_title('Error Rate by Chart Type', fontweight='bold')
        ax1.set_xlim([0, 1])
        for i, v in enumerate(df_errors_sorted['error_rate']):
            ax1.text(v + 0.02, i, f'{v:.1%}', va='center')
    
    # Chart 2: Score distribution
    all_scores = [m.get('score', 0) for m in run_results['per_chart'].values()]
    ax2.hist(all_scores, bins=20, color='steelblue', alpha=0.8, edgecolor='black')
    ax2.set_xlabel('Score')
    ax2.set_ylabel('Frequency')
    ax2.set_title('Score Distribution (All Charts)', fontweight='bold')
    ax2.axvline(np.mean(all_scores), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(all_scores):.3f}')
    ax2.legend()
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nScore Statistics:")
    print(f"  Mean: {np.mean(all_scores):.3f}")
    print(f"  Median: {np.median(all_scores):.3f}")
    print(f"  Std Dev: {np.std(all_scores):.3f}")
    print(f"  Min: {np.min(all_scores):.3f}")
    print(f"  Max: {np.max(all_scores):.3f}")
else:
    print("Insufficient data for error category analysis")

In [ ]:
# Failure reason breakdown
if 'per_chart' in run_results:
    failure_reasons = {}
    
    for chart_id, metrics in run_results['per_chart'].items():
        if metrics.get('score', 1.0) < 0.5:  # Only failed predictions
            reason = metrics.get('failure_reason', 'unknown')
            failure_reasons[reason] = failure_reasons.get(reason, 0) + 1
    
    if failure_reasons:
        df_reasons = pd.DataFrame(list(failure_reasons.items()), columns=['Reason', 'Count'])
        df_reasons = df_reasons.sort_values('Count', ascending=False)
        
        print("Failure Reasons (Failed Predictions Only):")
        print("=" * 50)
        print(df_reasons.to_string(index=False))
        
        # Pie chart
        fig, ax = plt.subplots(figsize=(10, 6))
        colors = plt.cm.Set3(range(len(df_reasons)))
        ax.pie(df_reasons['Count'], labels=df_reasons['Reason'], autopct='%1.1f%%',
               colors=colors, startangle=90)
        ax.set_title('Failure Reason Distribution', fontweight='bold', fontsize=12)
        plt.tight_layout()
        plt.show()
    else:
        print("No failures found in this run")
else:
    print("No per-chart data available")

## Summary and Actionable Findings

### Key Observations
1. **Top Error Sources:** Identify which chart types have highest error rates
2. **Score Distribution:** Majority of predictions fall into specific score ranges
3. **Failure Patterns:** Common failure reasons across different chart types

### Recommendations
1. Focus improvement efforts on high-error chart types
2. Investigate root causes of top failure reasons
3. Consider retraining with harder examples from failing categories
4. Implement specific handling for problematic chart types

### Next Steps
- Review sample failed charts in detail
- Compare against successful examples
- Implement targeted fixes for identified issues